# ADAC Workshop 3 (26L) - Dataproc clusters and Spark job scaling

In Lab 2 you built a SparkML pipeline on GUIDE using a managed Spark session. In this lab you switch to a **standard Dataproc cluster** and submit PySpark jobs with explicit Spark properties.

The goal is not only to run Spark, but to observe what changes when you allocate more executors to the same job.

#### Student tasks

- Create a standard Dataproc cluster in your coupon project.
- Submit the same PySpark job with 1, 2, and 4 executors.
- Compare runtime and explain why speedup is not perfectly linear.
- Turn your Lab 2 ML pipeline into a submitted PySpark job. You write that implementation yourself.

#### Cost

A standard Dataproc cluster keeps billing while it exists. Use the auto-delete flag below, and delete the cluster manually at the end of the lab.

## Part 0 - Setup

Use a **Python 3** notebook kernel for this lab. Do not keep a PySpark remote kernel running while you submit jobs, because an interactive PySpark shell can occupy YARN resources and make submitted jobs wait.

You will use the same GCS data layout as Lab 2:

```text
gs://adac-26l-YOUR_ID/guide/GUIDE_Train.csv
gs://adac-26l-YOUR_ID/guide/GUIDE_Test.csv
```

In [ ]:
PROJECT_ID = ""   # TODO: your coupon project id, e.g. "adac-lab-2-2026"
USER_ID = ""      # TODO: your Google username, lowercase; same id as in Lab 2 bucket

REGION = "europe-west1"
ZONE = "europe-west1-b"
CLUSTER = f"adac-lab3-{USER_ID}"
BUCKET = f"adac-26l-{USER_ID}"

TRAIN = f"gs://{BUCKET}/guide/GUIDE_Train.csv"
TEST = f"gs://{BUCKET}/guide/GUIDE_Test.csv"

print("project:", PROJECT_ID)
print("cluster:", CLUSTER)
print("train:", TRAIN)
print("test:", TEST)

### 0.1 Verify project and data

This should list both GUIDE CSV files in your bucket. If it fails, finish Lab 2 Part 0.4 first.

In [ ]:
!gcloud config set project {PROJECT_ID}
!gsutil ls -lh {TRAIN} {TEST}

## Part 1 - Create a standard Dataproc cluster

For this lab, keep the cluster simple and visible:

- 1 master node,
- 4 worker nodes,
- `n4-standard-4` machines,
- no GPU,
- Lightning engine off,
- Component Gateway enabled for YARN/Spark web interfaces. The Jupyter optional component is not needed for this submitted-job lab.

The important learning point is the relation between **workers**, **YARN resources**, and **Spark executors**. You will control executor count at job-submit time.

In [ ]:
!gcloud services enable dataproc.googleapis.com compute.googleapis.com storage.googleapis.com --project={PROJECT_ID}

In [ ]:
!gcloud dataproc clusters create {CLUSTER} \
  --project={PROJECT_ID} \
  --region={REGION} \
  --zone={ZONE} \
  --master-machine-type=n4-standard-4 \
  --worker-machine-type=n4-standard-4 \
  --num-workers=4 \
  --master-boot-disk-size=100GB \
  --worker-boot-disk-size=100GB \
  --image-version=2.2-debian12 \
  --enable-component-gateway \
  --delete-max-idle=2h \
  --properties=spark:spark.executor.cores=1,spark:spark.executor.memory=2g,spark:spark.executor.instances=2

### 1.1 Check cluster and YARN worker state

Dataproc may show the cluster as `RUNNING` before you have looked at the resource manager. Use YARN to verify that all worker NodeManagers are registered.

In [ ]:
!gcloud dataproc clusters describe {CLUSTER} \
  --project={PROJECT_ID} --region={REGION} \
  --format='table(status.state,config.workerConfig.numInstances,config.softwareConfig.properties.spark:spark.executor.instances)'

!gcloud compute ssh {CLUSTER}-m \
  --project={PROJECT_ID} --zone={ZONE} \
  --command='yarn node -list -all && echo === apps === && yarn application -list'

## Part 2 - Smoke job: read, count, group

Before running the full ML pipeline, run a small, reliable job that proves the cluster can read GUIDE from GCS and distribute work across executors.

This job does exactly three actions:

- count Train rows,
- count Test rows,
- group Train by `IncidentGrade`.

It also prints how many executors were actually allocated.

In [ ]:
%%writefile guide_scale_smoke.py
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


TRAIN_PATH = "gs://adac-26l-REPLACE_ME/guide/GUIDE_Train.csv"
TEST_PATH = "gs://adac-26l-REPLACE_ME/guide/GUIDE_Test.csv"


def executor_snapshot(spark, label):
    memory_status = spark.sparkContext._jsc.sc().getExecutorMemoryStatus()
    total_entries = memory_status.size()
    executors = max(total_entries - 1, 0)
    print(f"\n[{label}] executor memory entries={total_entries}; executors excluding driver={executors}")
    print(f"  {memory_status}")


def main():
    spark = SparkSession.builder.appName("guide-scale-smoke").getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    conf = spark.sparkContext.getConf()
    print("\nSpark configuration")
    for key in [
        "spark.master",
        "spark.dynamicAllocation.enabled",
        "spark.executor.instances",
        "spark.executor.cores",
        "spark.executor.memory",
    ]:
        print(f"  {key}={conf.get(key, '<unset>')}")

    executor_snapshot(spark, "before read")

    start = time.time()
    train = spark.read.option("header", "true").csv(TRAIN_PATH)
    test = spark.read.option("header", "true").csv(TEST_PATH)

    print("\nInput")
    print(f"  train partitions={train.rdd.getNumPartitions()}")
    print(f"  test partitions={test.rdd.getNumPartitions()}")

    train_count = train.count()
    test_count = test.count()
    print("\nCounts")
    print(f"  GUIDE_Train rows={train_count}")
    print(f"  GUIDE_Test rows={test_count}")

    print("\nGroup by IncidentGrade")
    train.groupBy("IncidentGrade").agg(F.count("*").alias("rows")).orderBy("IncidentGrade").show(50, truncate=False)

    executor_snapshot(spark, "after actions")
    print(f"\nElapsed seconds={time.time() - start:.2f}")
    spark.stop()


if __name__ == "__main__":
    main()


Patch the script so it uses **your** bucket. This keeps the submitted job self-contained and easy to inspect.

In [ ]:
from pathlib import Path

p = Path("guide_scale_smoke.py")
p.write_text(p.read_text().replace("gs://adac-26l-REPLACE_ME", f"gs://{BUCKET}"))
print(p.read_text().splitlines()[0:8])

### 2.1 Run with 1, 2, and 4 executors

Each run uses the same cluster and the same job script. Only Spark job properties change.

We disable dynamic allocation because this lab is about a controlled experiment: exactly 1, 2, or 4 executors.

In [ ]:
import re
import subprocess
import time

RESULTS = []

for executors in [1, 2, 4]:
    props = ",".join([
        "spark.dynamicAllocation.enabled=false",
        f"spark.executor.instances={executors}",
        "spark.executor.cores=1",
        "spark.executor.memory=2g",
    ])
    cmd = [
        "gcloud", "dataproc", "jobs", "submit", "pyspark", "guide_scale_smoke.py",
        f"--cluster={CLUSTER}",
        f"--region={REGION}",
        f"--project={PROJECT_ID}",
        f"--properties={props}",
    ]

    print("\n" + "=" * 80)
    print(f"Submitting smoke job with {executors} executor(s)")
    print("$", " ".join(cmd))

    t0 = time.perf_counter()
    proc = subprocess.run(cmd, text=True, capture_output=True)
    submit_wall_seconds = time.perf_counter() - t0

    output = proc.stdout + "\n" + proc.stderr
    print(output)

    elapsed_match = re.search(r"Elapsed seconds=([0-9.]+)", output)
    job_match = re.search(r"Job \[([^\]]+)\] submitted", output)
    allocated_match = re.findall(r"executors excluding driver=([0-9]+)", output)

    RESULTS.append({
        "executors_requested": executors,
        "executors_observed_after_actions": int(allocated_match[-1]) if allocated_match else None,
        "job_elapsed_seconds": float(elapsed_match.group(1)) if elapsed_match else None,
        "submit_wall_seconds": submit_wall_seconds,
        "job_id": job_match.group(1) if job_match else None,
        "returncode": proc.returncode,
    })

    if proc.returncode != 0:
        raise RuntimeError(f"Job failed for executors={executors}")

RESULTS

### 2.2 Plot and interpret scaling

You should see speedup, but not perfect speedup. The job still pays for scheduling, GCS reads, driver work, and stage boundaries.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_df = pd.DataFrame(RESULTS)
display(results_df)

ax = results_df.plot(x="executors_requested", y="job_elapsed_seconds", marker="o", legend=False)
ax.set_xlabel("executors requested")
ax.set_ylabel("job elapsed seconds")
ax.set_title("Strong scaling: same job, more executors")
ax.grid(True)
plt.show()

### 2.3 Student answers

Fill these in after the three runs.

In [ ]:
from IPython.display import Markdown, display

PROMPT = "How much faster was 2 executors vs 1 executor? How much faster was 4 vs 2?"
ANSWER = ""  # TODO
display(Markdown("**" + PROMPT + "**\n\n" + ANSWER))

In [ ]:
PROMPT = "Why was the 4-executor run not twice as fast as the 2-executor run?"
ANSWER = ""  # TODO: mention GCS IO, scheduling overhead, non-parallel driver work, and limited partitions/stages if relevant
display(Markdown("**" + PROMPT + "**\n\n" + ANSWER))

## Part 3 - Optional: scale the cluster itself

The previous section changed only the number of executors requested by a job. You can also change the number of worker VMs in the cluster.

Run cluster scaling only when no jobs are active.

In [ ]:
# Scale down to 2 workers.
# !gcloud dataproc clusters update {CLUSTER} \
#   --project={PROJECT_ID} --region={REGION} \
#   --num-workers=2 --graceful-decommission-timeout=2m

# Scale back to 4 workers.
# !gcloud dataproc clusters update {CLUSTER} \
#   --project={PROJECT_ID} --region={REGION} \
#   --num-workers=4

## Part 4 - Student task: submit your Lab 2 ML pipeline as a job

Now turn the notebook pipeline from Lab 2 into a PySpark script and submit it to the same cluster.

Do **not** copy notebook-only display code. A submitted job should:

- read Train and Test paths from command-line arguments,
- build the same `StringIndexer -> OneHotEncoder -> VectorAssembler -> classifier` pipeline from Lab 2,
- fit on Train,
- transform Test,
- print F1 and accuracy using `MulticlassClassificationEvaluator`,
- print elapsed seconds,
- stop Spark at the end.

The skeleton uses `TRAIN_LIMIT = 50_000` and `TEST_LIMIT = 20_000`, reads only the pipeline columns, and avoids `inferSchema` so the first job finishes quickly. After the script works, remove or increase the limits for the full-data experiment.

The cell below is a skeleton. You must fill in the TODOs from your Lab 2 solution.

In [ ]:
%%writefile guide_ml_pipeline_job.py
import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# TODO: import the SparkML stages/classes you used in Lab 2.


CAT_COLS = ["Category", "EntityType", "EvidenceRole", "CountryCode"]
USE_COLS = ["IncidentGrade", *CAT_COLS]
TRAIN_LIMIT = 50_000
TEST_LIMIT = 20_000


def build_pipeline():
    # TODO: paste/adapt your Lab 2 pipeline stages here.
    # Required output columns: features, label.
    # Required classifier: any multiclass-capable classifier from your Lab 2 work.
    raise NotImplementedError("Implement your Lab 2 SparkML pipeline here")


def read_labeled_subset(spark, path, limit):
    return (
        spark.read.option("header", "true").csv(path)
        .select(*USE_COLS)
        .where(F.col("IncidentGrade").isNotNull())
        .limit(limit)
        .cache()
    )


def main(train_path, test_path):
    spark = SparkSession.builder.appName("guide-ml-pipeline-job").getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    start = time.time()
    train = read_labeled_subset(spark, train_path, TRAIN_LIMIT)
    test = read_labeled_subset(spark, test_path, TEST_LIMIT)
    print("train rows", train.count())
    print("test rows", test.count())

    pipeline = build_pipeline()
    model = pipeline.fit(train)
    pred = model.transform(test)

    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    print("f1", evaluator_f1.evaluate(pred))
    print("accuracy", evaluator_acc.evaluate(pred))
    print(f"Elapsed seconds={time.time() - start:.2f}")
    spark.stop()


if __name__ == "__main__":
    if len(sys.argv) != 3:
        raise SystemExit("Usage: guide_ml_pipeline_job.py TRAIN_PATH TEST_PATH")
    main(sys.argv[1], sys.argv[2])


After you implement `guide_ml_pipeline_job.py`, submit it exactly like the smoke job. Start small, then scale.

In [ ]:
# Do not run this until you have implemented guide_ml_pipeline_job.py.
#
# for executors in [1, 2, 4]:
#     props = ",".join([
#         "spark.dynamicAllocation.enabled=false",
#         f"spark.executor.instances={executors}",
#         "spark.executor.cores=1",
#         "spark.executor.memory=2g",
#     ])
#     !gcloud dataproc jobs submit pyspark guide_ml_pipeline_job.py \
#       --cluster={CLUSTER} --region={REGION} --project={PROJECT_ID} \
#       --properties={props} -- {TRAIN} {TEST}

### 4.1 Student answers

Record your ML job results.

In [ ]:
PROMPT = "Did more executors change model quality, runtime, or both? Explain."
ANSWER = ""  # TODO
display(Markdown("**" + PROMPT + "**\n\n" + ANSWER))

## Part 5 - Teardown

Delete the cluster when you are done. Keeping a standard Dataproc cluster alive keeps charging your coupon.

The GCS bucket is the same bucket as Lab 2. Delete it only if you are sure you no longer need the data.

In [ ]:
!gcloud dataproc clusters delete {CLUSTER} \
  --project={PROJECT_ID} --region={REGION} --quiet

# Optional: delete your GUIDE data bucket too.
# !gsutil -m rm -r gs://{BUCKET}